In [ ]:
# Task 1 & 2: Create transforms
import torchvision.transforms as transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# Task 3: Create DataLoaders
from torch.utils.data import DataLoader
import torchvision

dataset_path = './images_dataSAT/'
full_dataset = torchvision.datasets.ImageFolder(root=dataset_path, transform=train_transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


# Task 4: Design and train CNN-ViT hybrid model
# (Simplified ViT implementation)
import torch.nn as nn

class CNNViTHybrid(nn.Module):
    def __init__(self, cnn_backbone, embed_dim=768, num_heads=12, depth=12, num_classes=2):
        super().__init__()
        self.cnn = cnn_backbone
        # Remove classifier from CNN
        self.cnn = nn.Sequential(*list(cnn_backbone.children())[:-1])
        
        # Transformer components
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=depth)
        
        self.classifier = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        cnn_features = self.cnn(x)
        cnn_features = cnn_features.flatten(2).permute(0, 2, 1)
        transformer_out = self.transformer(cnn_features)
        out = transformer_out.mean(dim=1)
        return self.classifier(out)

# model_test = CNNViTHybrid(pretrained_resnet, embed_dim=768, num_heads=12, depth=12)
# Train for epochs=5


# Task 5 & 6: Compare performance and training times
# Plot validation loss comparison and training time comparison